# Fine-tune bengaliai-asr-whisper-medium — LoRA on Kaggle T4

**Read this before running.** This notebook is Layer 5 of the Omnicare STT plan. It is deliberately marked *conditional*: you only run it if Layer 2's `evaluate.py` shows the baseline is not good enough for medical use. On the last measured baseline (13.2% WER FLEURS, 3.1% WER synthetic medical), fine-tuning is a wasteful improvement over a model that is already close to ceiling for the data we have.

**When you should run it**
- You have collected at least 5–10 hours of *real* domain audio (patient consults, mock consultations, or your own voice through the production mic) with matching transcripts.
- The current model's WER on that real-domain audio is >10%.
- You have accepted Bengali.AI Speech Recognition competition rules on Kaggle so the dataset is attachable.

**When NOT to run it**
- Only source of medical data is TTS-synthesized. Fine-tuning on synthetic will teach the model TTS quirks, not real acoustic conditions.
- You have no medical audio at all — fine-tuning on Bengali.AI data alone would re-teach the model what it already knows (the base model was trained on this same corpus).

**What this notebook produces**
- A LoRA adapter (~50 MB) saved to `/kaggle/working/lora-adapter/`
- A merged full-weight checkpoint (~3 GB) saved to `/kaggle/working/merged-model/`
- A WER/CER report against FLEURS bn_in for regression check

**What you do with it after**
1. Download `merged-model/` from Kaggle.
2. Copy into `new-stt/models/bengaliai-asr-whisper-medium-ft/`.
3. Edit `new-stt/config.py` — change `STT_MODEL_DIR` to point at the new folder. Nothing else in the codebase moves.
4. Re-run `python evaluate.py` to confirm the fine-tune actually helped on your eval set.

**Prerequisites in Kaggle UI (before starting a session)**
1. Enable GPU: Settings → Accelerator → **GPU T4 × 2**
2. Enable internet: Settings → Internet → **On** (needed for HF hub + FLEURS)
3. Add data: **+ Add Data** → search *Bengali.AI Speech Recognition* → Add
4. Add your own real-audio dataset the same way (upload it to Kaggle as a private dataset first)

## 1. Install extras

Kaggle images ship with `transformers` and `torch`; we add `peft` (LoRA), `accelerate` (multi-GPU training loop), `jiwer` (WER/CER), and the `datasets` upgrades that current transformers expects.

In [ ]:
!pip install -q -U 'transformers>=4.45' 'accelerate>=0.30' 'peft>=0.11' 'datasets>=3.0' jiwer soundfile librosa

## 2. Configuration — every knob in one cell

Same principle as `config.py` in the main project: constants at the top so re-runs with different settings are one-line edits.

In [ ]:
from pathlib import Path

# --- Base model ---
# tugstugi's fine-tune is on HF hub as this repo. If it gets 404-gated (has happened before),
# swap to `openai/whisper-medium` — you lose the Bangla priors and need much more data.
BASE_MODEL = 'tugstugi/bengaliai-asr-whisper-medium'

# --- Kaggle inputs ---
BENGALI_AI_DIR = Path('/kaggle/input/bengaliai-speech')
TRAIN_CSV      = BENGALI_AI_DIR / 'train.csv'
TRAIN_MP3_DIR  = BENGALI_AI_DIR / 'train_mp3s'

# YOUR real-domain dataset. Replace 'my-medical-audio' with whatever you named your Kaggle dataset.
# Expected layout inside that folder: a manifest CSV with columns (id, sentence) + a folder of wav/mp3.
MEDICAL_DIR     = Path('/kaggle/input/my-medical-audio')
MEDICAL_CSV     = MEDICAL_DIR / 'manifest.csv'
MEDICAL_AUDIO_DIR = MEDICAL_DIR / 'audio'

# --- Training scale ---
# Small numbers on purpose. LoRA + a small mix converges fast; scale up only if regression eval
# says the adapter is undertrained. Bigger = slower (and Kaggle sessions are 12h capped).
BENGALI_AI_SAMPLE_HOURS = 30      # ~5000 clips, general-Bangla regularization
MEDICAL_MULTIPLIER      = 5       # duplicate medical clips this many times so they aren't drowned out
MAX_AUDIO_SECONDS       = 30      # Whisper's window; longer clips are dropped
TRAIN_BATCH_SIZE        = 8       # per GPU (T4 has 16GB — this fits with LoRA + fp16)
EVAL_BATCH_SIZE         = 8
LEARNING_RATE           = 1e-4    # standard LoRA LR; base-model full-FT would use ~1e-5
NUM_EPOCHS              = 3       # 3 passes over the mixed set; watch eval loss to decide

# --- LoRA ---
LORA_R       = 16                 # adapter rank; 8-32 is the useful range
LORA_ALPHA   = 32                 # scaling; conventional to set = 2×r
LORA_DROPOUT = 0.05

# --- Outputs ---
OUT_DIR      = Path('/kaggle/working')
ADAPTER_DIR  = OUT_DIR / 'lora-adapter'
MERGED_DIR   = OUT_DIR / 'merged-model'

for d in (ADAPTER_DIR, MERGED_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 3. Load base model + processor

`WhisperProcessor` handles feature extraction (mel spectrogram) and tokenization. `WhisperForConditionalGeneration` is the seq2seq model itself. We load in fp16 to halve VRAM — LoRA adapters stay fp32 so training numerics are still fine.

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language='bn', task='transcribe')

model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,   # halves memory; training grads still fp32 via mixed precision
    device_map='auto',           # T4×2 → sharded across both GPUs automatically
)

# Whisper's generation config has forced_decoder_ids that lock the language. For fine-tuning
# we clear those (the model relearns language from data), but keep them set for generation-time
# eval below.
model.config.forced_decoder_ids  = None
model.config.suppress_tokens     = []

print(f'model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

## 4. Attach LoRA adapter

Instead of training all 750M params, we freeze the base weights and insert small trainable rank-`r` matrices into every attention projection (`q_proj`, `k_proj`, `v_proj`, `out_proj`). Trainable params drop from 750M → ~15M. That's what makes this fit on a T4.

Trade-off: LoRA-fine-tuned models are ~1-2% WER worse than a full fine-tune on the same data, but need ~10× less GPU memory and are 3-5× faster to train.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
    task_type='SEQ_2_SEQ_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # expect: trainable=~15M / total=~765M

## 5. Build the training dataset

Two sources, mixed:
- Bengali.AI sample — general Bangla regularization. Without this, LoRA overfits your small medical set and forgets how to transcribe general speech.
- Your medical clips — the actual thing you're trying to improve on. Duplicated `MEDICAL_MULTIPLIER` times so they aren't drowned out by the 30-hour Bengali.AI sample.

The `prepare` function converts each row into the tensors Whisper wants: log-mel input features + tokenized target text.

In [ ]:
import pandas as pd
from datasets import Dataset, Audio, concatenate_datasets

# --- Bengali.AI Speech sample ---
ba_df = pd.read_csv(TRAIN_CSV)
ba_df = ba_df[ba_df['split'] == 'train']
# Sample ~30h. Average clip is ~7s → 30h × 3600s / 7s ≈ 15k clips. Halve for safety with the T4 clock.
ba_df = ba_df.sample(n=int(BENGALI_AI_SAMPLE_HOURS * 3600 / 7 / 2), random_state=42)
ba_df['audio'] = ba_df['id'].apply(lambda i: str(TRAIN_MP3_DIR / f'{i}.mp3'))
ba_df = ba_df[['audio', 'sentence']]

# --- Medical set ---
if MEDICAL_CSV.exists():
    med_df = pd.read_csv(MEDICAL_CSV)
    med_df['audio'] = med_df['id'].apply(lambda i: str(MEDICAL_AUDIO_DIR / i))
    med_df = med_df[['audio', 'sentence']]
    # Duplicate to give it real weight in the mix.
    med_df = pd.concat([med_df] * MEDICAL_MULTIPLIER, ignore_index=True)
else:
    print('No medical dataset attached — training on Bengali.AI only. This is a re-teach, not an improvement.')
    med_df = None

combined_df = pd.concat([ba_df, med_df]) if med_df is not None else ba_df
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle

train_ds = Dataset.from_pandas(combined_df).cast_column('audio', Audio(sampling_rate=16_000))
print(f'training clips: {len(train_ds):,}')

In [ ]:
def prepare(batch):
    audio = batch['audio']
    # Extract log-mel features (what Whisper actually eats).
    batch['input_features'] = processor.feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    # Tokenize the target transcript.
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names, num_proc=2)

# Drop anything longer than MAX_AUDIO_SECONDS (Whisper's window) — they'd be truncated anyway.
MAX_INPUT_LENGTH = MAX_AUDIO_SECONDS * 100   # log-mel is ~100 frames/sec
train_ds = train_ds.filter(lambda x: len(x['input_features'][0]) <= MAX_INPUT_LENGTH)
print(f'after length filter: {len(train_ds):,}')

## 6. Data collator + training arguments

The collator pads input features and labels to the batch max separately (different lengths). Whisper wants `-100` as the padding label so the loss function ignores those positions.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class WhisperDataCollator:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split then pad audio and labels separately — they have different shapes.
        input_feats = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_feats, return_tensors='pt')

        label_feats = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # Whisper prepends BOS in decoder; drop it from labels to avoid double-BOS.
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = WhisperDataCollator(processor=processor)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR / 'trainer'),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=2,      # effective batch = 8 × 2 × 2 GPUs = 32
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,                          # T4 has hw fp16, saves VRAM + time
    save_strategy='epoch',
    logging_steps=25,
    report_to='none',                   # no wandb/tensorboard by default
    remove_unused_columns=False,        # trainer would otherwise strip input_features
    label_names=['labels'],
    predict_with_generate=False,        # LoRA + generate has known compat bugs; eval separately
)

trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=train_ds,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

## 7. Train

Expect ~2–4 hours on Kaggle T4×2 for the default 30h+medical mix, 3 epochs. If the session is about to hit the 12h limit, checkpoints are saved per epoch so you can resume.

In [ ]:
trainer.train()

## 8. Save the LoRA adapter and a merged full-weight checkpoint

Two artifacts:
- **Adapter** (`~50 MB`) — tiny, HF-uploadable, requires base model at load time.
- **Merged** (`~3 GB`) — self-contained, drop-in replacement for the base model. Use this in the main project.

In [ ]:
# Adapter — small, quick to save.
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

# Merged — combines the LoRA delta back into base weights for standalone use.
merged = model.merge_and_unload()
merged.save_pretrained(str(MERGED_DIR))
processor.save_pretrained(str(MERGED_DIR))

!ls -lh {ADAPTER_DIR} {MERGED_DIR}

## 9. Regression check — WER/CER on FLEURS bn_in

We evaluate the merged model on public FLEURS to confirm we haven't destroyed general Bangla. Use the same `bnorm` normalization as the main project so numbers are comparable to what `evaluate.py` prints locally.

In [ ]:
# Inline copy of new-stt/bnorm.py so this notebook is self-contained.
import re, unicodedata
_PUNCT = '।॥,;:!?"\'`()[]{}<>«»“”‘’…–—./\\|@#$%^&*_+=~'
_PUNCT_RE = re.compile(f'[{re.escape(_PUNCT)}]')
_DASH_RE  = re.compile(r'[-‐‑]')
_ZW_RE    = re.compile('[\u200b\u200c\u200d\ufeff]')
_WS_RE    = re.compile(r'\s+')
_DIGIT_MAP = {ord(b): str(i) for i, b in enumerate('০১২৩৪৫৬৭৮৯')}
def bnorm(t):
    t = unicodedata.normalize('NFC', t)
    t = _ZW_RE.sub('', t); t = _DASH_RE.sub(' ', t); t = _PUNCT_RE.sub('', t)
    return _WS_RE.sub(' ', t.translate(_DIGIT_MAP)).strip()

In [ ]:
from datasets import load_dataset
import jiwer
from transformers import pipeline

asr = pipeline(
    'automatic-speech-recognition',
    model=str(MERGED_DIR),
    device=0,
    chunk_length_s=30, stride_length_s=(6, 0),
    return_timestamps=False,
)

fleurs = load_dataset('google/fleurs', 'bn_in', split='test', streaming=True)

refs, hyps = [], []
for i, row in enumerate(fleurs):
    if i >= 40:
        break
    hyp = asr({'array': row['audio']['array'], 'sampling_rate': row['audio']['sampling_rate']})['text']
    refs.append(bnorm(row['transcription']))
    hyps.append(bnorm(hyp))

print(f'FLEURS bn_in ({len(refs)} clips):')
print(f'  WER = {jiwer.wer(refs, hyps)*100:.1f}%')
print(f'  CER = {jiwer.cer(refs, hyps)*100:.1f}%')
print('\nCompare to baseline in Omnicare memory: 13.2% WER / 3.2% CER.')
print('If the fine-tune shows worse FLEURS numbers, the model has drifted toward your\n'
      'medical domain (expected). Check the medical eval to see if the trade paid off.')

## 10. Take the checkpoint home

**Download from Kaggle UI**: right sidebar → `/kaggle/working/merged-model/` → download as a `.zip`.

**Local drop-in** (on your Mac):
```bash
cd ~/claude/new-stt
mkdir -p models/bengaliai-asr-whisper-medium-ft
unzip ~/Downloads/merged-model.zip -d models/bengaliai-asr-whisper-medium-ft
```

Then in `config.py` change the one line:
```python
STT_MODEL_DIR = MODELS_DIR / 'bengaliai-asr-whisper-medium-ft'
STT_MODEL_NAME = 'bengaliai-asr-whisper-medium-ft'
```

Re-run `python evaluate.py` on your local eval set (FLEURS + medical) and confirm the fine-tune actually beat the baseline on the audio you care about. If it didn't, revert `config.py` — the base model was better.